# Multilingual Keyword Spotting — Embedding-First (v3, bigger backbone)

**Course**: Intro to Deep Learning &nbsp;|&nbsp; **Due**: May 14, 2026 (with extension)

## What changed from v2

v2 hit a capacity ceiling: 296K params on 450 classes ≈ 660 params/class, vs. Zhang 2017's 3,800/class baseline. Val Top-1 plateaued at ~14%, AUC 0.905, F1 0.38 — embedding worked for 5-shot detection but Top-1 was capacity-bound.

**v3 widens the DS-CNN backbone to ~~1.55M params** (still well inside the TinyML envelope: fits on ESP32-S3 with PSRAM at ~300ms latency). Everything else is identical to v2 — same data pipeline, same caching, same evaluation protocol, same export path.

### Architecture changes (cell 16 only)

| | v2 | v3 |
|---|---|---|
| First conv channels | 48 | **96** |
| DS blocks | 6 | **7** (wider throughout, plus one extra at 768ch) |
| Backbone width progression | 48→64→128→192→256 | **96→192→320→512→640→768** |
| Project layer in | 256 | **768** |
| Total params | 296K | **~1.55M (5× v2)** |

### Why this is the right size
- 1M+ params puts us in Mazumder-Small accuracy class (~30-40% Top-1 expected, AUC 0.94+)
- Still 5-6× smaller than Mazumder's 11M EfficientNet-B0
- int8-quantized footprint ~500KB → comfortably in ESP32-S3 PSRAM, ~300ms inference

### Other v3 changes (cell 4, 14, 16)
| | v2 | v3 |
|---|---|---|
| SAMPLES_PER_CLASS | 400 | **2000** (need to re-run fetch_kws_data.py with new cache dir) |
| EPOCHS | 30 | **50** (longer cosine anneal for bigger model) |
| WEIGHT_DECAY | 1e-4 | **5e-4** (stronger reg for 5x more params) |
| LABEL_SMOOTH | 0.1 | **0.05** (less needed at scale) |
| Train sampler | WeightedRandomSampler | **shuffle=True** (natural distribution; rare classes ride on cross-lingual transfer) |
| Classifier dropout | none | **Dropout(0.2)** before classifier |

### Cache routing
CACHE_DIR is now `kws_cache_v3` to keep v2 caches untouched. Re-run `fetch_kws_data.py` once with the new cache path (the cell below the install does it).

### Pipeline (unchanged from v2)

```
MSWC real audio (9 langs × top-50 keywords each)
        │
        ▼
DS-CNN classifier  (~~1.55M params, 256-dim penultimate)
        │
        ▼  freeze
multilingual embedding  ──▶  Section 7: t-SNE + silhouette diagnostics
        │
        ▼
5-shot head per target keyword (Section 8: ~120 head models; Section 8b TTS proxy; Section 8c novel-keyword demo)
        │
        ▼
F1 distribution + AUC per language  (Mazumder Fig. 2 reproduction)
```

### Key references
- Mazumder et al., *Interspeech 2021* — multilingual embedding + few-shot heads
- Mazumder et al., *NeurIPS 2021* — MSWC dataset
- Lin et al., *ICASSP 2020* — limits of synthetic speech for KWS
- Zhang et al., 2017 — DS-CNN ("Hello Edge") backbone
- Lei et al., *TASLP 2023* — embedding-dim sweep informed our 256-dim choice


In [ ]:
# ── Install pinned dependencies (Colab) ───────────────────────────────────────
# NO tensorflow, NO onnx2tf, NO qnnpack. Quantization/export live in the
# separate `multilingual_kws_export.ipynb` notebook.
# huggingface_hub: direct HF shard access (load_dataset is blocked for this
# dataset due to HF trust_remote_code policy changes).
%pip install -q "torchaudio>=2.0" "scikit-learn>=1.3" "seaborn>=0.13" "huggingface_hub>=0.20"
# After the first run on a fresh Colab: Runtime > Restart, then skip this cell.

In [9]:
import os
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab and not os.path.exists('/content/polyglot-kws'):
        !git clone https://github.com/verilog-indeed/polyglot-kws /content/polyglot-kws

In [10]:
import os, io, json, random, warnings, math, time, tempfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, roc_curve, auc
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# cudnn picks the fastest conv algorithm for fixed-size inputs (49x40).
# Safe to enable since our spec shape never changes.
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# No HuggingFace I/O happens in this notebook — fetch_kws_data.py does
# all dataset downloading and feature extraction.  We only load .npy files
# from CACHE_DIR (Drive in Colab, local dir otherwise).

print('Imports OK. Device:', device)


Imports OK. Device: cpu


In [ ]:
# Configuration
DEBUG = False   # flip to True for a tiny smoke-test run

LANGUAGES = ['en', 'de', 'fr', 'es', 'fa', 'ar']
LANG_NAMES = {
    'en': 'English', 'de': 'German',  'fr': 'French',
    'es': 'Spanish', 'fa': 'Persian', 'ar': 'Arabic',
}
# v3 typological set: 2 Germanic, 2 Romance, 1 Indo-Iranian, 1 Afro-Asiatic.
# Dropped from v2: ca, it, nl, rw (Romance/Germanic redundancy + edge-tts gap for rw).

# Audio / spectrogram - 49x40 log-mel, matching TF Lite Micro microfrontend.
# center=False + n_fft=win_length=640 gives exactly 49 frames:
#   1 + floor((16000 - 640) / 320) = 49
SAMPLE_RATE = 16_000
N_MELS      = 40
N_FFT       = 640
WIN_LENGTH  = 640
HOP_LENGTH  = 320
F_MIN       = 20
F_MAX       = 8_000

# Embedding training
TOP_K_PER_LANG       = 50
SAMPLES_PER_CLASS    = 2000  # cap per (lang, word); rare-language words plateau at availability
NOISE_FRACTION       = 0.10  # used only to size noise_pool for Section 8 heads
EMBEDDING_DIM        = 256
N_HELDOUT_KEYWORDS   = 20

# Training
BATCH_SIZE     = 128
EPOCHS         = 50
LR             = 3e-3
WEIGHT_DECAY   = 5e-4
LABEL_SMOOTH   = 0.05

# 5-shot head
N_SHOT             = 5
N_UNKNOWN_PER_HEAD = 128
N_NOISE_PER_HEAD   = 25
HEAD_EPOCHS        = 40
HEAD_LR            = 1e-2
F1_THRESHOLD       = 0.8

# ── Run-mode switch ──────────────────────────────────────────────────────────
# 'auto'   : load cache if present, otherwise compute fresh and save
# 'fresh'  : always recompute (overwrites caches)
# 'cached' : require cache (FileNotFoundError if missing)
RUN_MODE          = 'auto'
TRAINING_RUN_MODE = RUN_MODE   # Section 6 embedding training
FEWSHOT_RUN_MODE  = RUN_MODE   # Section 8 5-shot sweep
TTS_RUN_MODE      = RUN_MODE   # Section 8b TTS-enrolled heads

# Debug overrides
if DEBUG:
    LANGUAGES            = ['en', 'de']
    TOP_K_PER_LANG       = 5
    SAMPLES_PER_CLASS    = 40
    N_HELDOUT_KEYWORDS   = 2
    BATCH_SIZE           = 32
    EPOCHS               = 3
    HEAD_EPOCHS          = 5
    N_UNKNOWN_PER_HEAD   = 20
    N_NOISE_PER_HEAD     = 5
    print('*** DEBUG MODE - small data, fast run, results not meaningful ***')

# Colab / local path setup
_cache_name = 'kws_cache_v3_debug' if DEBUG else 'kws_cache_v3'

if _in_colab:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT_DIR = '/content/drive/MyDrive/'
    REPO_DIR = '/content/polyglot-kws/'
    SHARD_DIR = '/content/kws_shards'
    print('Colab detected - cache routed to Google Drive.')
else:
    ROOT_DIR = './'
    REPO_DIR = './'
    SHARD_DIR = './kws_shards'
    print('Local run.')

CACHE_DIR = Path(f'{ROOT_DIR}/{_cache_name}')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATS_DIR   = CACHE_DIR / 'feats'
INVENT_PATH = CACHE_DIR / 'keyword_inventory.json'
CKPT_PATH   = CACHE_DIR / 'embedding_best.pt'
FEATS_DIR.mkdir(exist_ok=True)

print('Cache at  :', CACHE_DIR.resolve())
print(f'Languages : {LANGUAGES}')
print(f'Classes   : up to {len(LANGUAGES) * TOP_K_PER_LANG}  (noise excluded from embedding training)')


> **Before running this notebook**, run `fetch_kws_data.py` once. The repo is already cloned by the cell above, so just run:
>
> ```bash
> # Upload mswc-metadata.json to the root of your Drive first, then:
> !python /content/polyglot-kws/fetch_kws_data.py \
>     --metadata /content/drive/MyDrive/mswc-metadata.json \
>     --root     /content/drive/MyDrive/kws_cache
> ```
>
The script downloads MSWC shards from HuggingFace, decodes audio, computes 49×40 log-mel features, and saves `feats/{kind}/{lang}/{word}.npy` + `keyword_inventory.json` to Drive. **The notebook never contacts HuggingFace** — it loads everything from Drive cache.
>
> For a quick smoke-test before the full run, add `--debug` (2 langs, top-5 words, 40 samples).

In [ ]:
!python {REPO_DIR}/fetch_kws_data.py \
--metadata {ROOT_DIR}/mswc-metadata.json \
--root {CACHE_DIR} \
--langs {' '.join(LANGUAGES)} \
--top-k {TOP_K_PER_LANG} \
--n-heldout {N_HELDOUT_KEYWORDS} \
--samples {SAMPLES_PER_CLASS} \
--shard-dir {SHARD_DIR} \
--split train

---
## Section 2 — MSWC keyword inventory

For each of the 9 languages we pick the top **50 most frequent** words (≥3 chars) from `mswc-metadata.json` wordcounts as our embedding training classes. We also reserve a **held-out keyword bank** of 20 words per language that the embedding never sees during training — these are the targets of the Section 8 5-shot evaluation.

The inventory is saved to `keyword_inventory.json` so subsequent runs are deterministic. The `build_inventory` function below reads this file directly from Drive; the HuggingFace streaming path is never triggered once the script has run.

**Why inventory-first**: MSWC is uneven across languages (English ~1957 hours, Persian ~7.6 hours). If low-resource languages can't supply enough heldout candidates, we want to know before training starts.

In [ ]:
# ── Load the keyword inventory built by fetch_kws_data.py ────────────────────
# The notebook never touches HuggingFace.  fetch_kws_data.py reads
# mswc-metadata.json, ranks each language's words by their true clip count
# (len(filenames[word])), and saves keyword_inventory.json + per-(lang, word)
# .npy feature files to CACHE_DIR.  Here we just load and sanity-check.

if not INVENT_PATH.exists():
    raise FileNotFoundError(
        f'No inventory at {INVENT_PATH}.'
        f'Run fetch_kws_data.py once before this notebook, e.g.:'
        f'  !python /content/polyglot-kws/fetch_kws_data.py \\'
        f'      --metadata /content/drive/MyDrive/mswc-metadata.json \\'
        f'No inventory at {INVENT_PATH}.'
        f'Run fetch_kws_data.py once before this notebook, e.g.:'
        f'  !python /content/polyglot-kws/fetch_kws_data.py \\'
        f'      --metadata /content/drive/MyDrive/mswc-metadata.json \\'
        f'      --root     {CACHE_DIR}'
        + ('  --debug' if DEBUG else '')
    )

inventory = json.loads(INVENT_PATH.read_text(encoding='utf-8'))

missing = [l for l in LANGUAGES if l not in inventory]
if missing:
    raise RuntimeError(
        f'Inventory is missing languages {missing}. '
        f'Re-run fetch_kws_data.py with --langs {" ".join(LANGUAGES)}.'
    )

print(f'Loaded inventory from {INVENT_PATH}')
print(f'{"lang":6s} {"#train":>7s} {"#heldout":>9s} {"min clips":>10s} {"max clips":>10s}')
print(f'{"lang":6s} {"#train":>7s} {"#heldout":>9s} {"min clips":>10s} {"max clips":>10s}')
for lang in LANGUAGES:
    inv     = inventory[lang]
    train   = inv.get('training', [])
    heldout = inv.get('heldout', [])
    counts  = inv.get('counts', {})
    tc = [counts.get(w, 0) for w in train]
    print(f'{lang:6s} {len(train):7d} {len(heldout):9d} '
          f'{(min(tc) if tc else 0):10d} {(max(tc) if tc else 0):10d}')


---
## Section 3 — Log-mel feature extraction

We convert each 1-second clip to a **49 × 40 log-mel spectrogram** (matching TF Lite Micro's microfrontend convention used by Mazumder et al., so our results are directly comparable to theirs).

**Pipeline**: waveform → STFT (40 ms window, 20 ms hop) → power → 40-band Mel filterbank → log → normalize to ≈ [0, 1].

**Why log-mel and not raw waveform / MFCCs**:
- **Raw waveform** has 16,000 numbers/sec with complex long-range dependencies. Hard for small CNNs to learn from.
- **MFCCs** apply a DCT on top of log-mel and discard high-order coefficients. The DCT decorrelates features — useful for GMM-HMM models, harmful for CNNs that can learn their own decorrelation given the full input. Modern KWS papers universally use log-mel.

**SpecAugment** masks small bands of time and frequency during training as a regularizer (Park et al. 2019). Applied only in train mode.

In [ ]:
# Mel transforms live on the same device as the training data so feature
# extraction during streaming runs on GPU when available.
#
# center=False is required to get exactly 49 frames:
#   frames = 1 + floor((16000 - n_fft) / hop_length)
#          = 1 + floor((16000 - 640)  / 320) = 49
# center=True (default) pads n_fft//2 on each side → 51 frames.
_mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
    power=2.0, center=False,
).to(device)
_amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80).to(device)

# SpecAugment for training-time regularization
spec_augment = nn.Sequential(
    T.FrequencyMasking(freq_mask_param=8),
    T.TimeMasking(time_mask_param=10),
)


def waveform_to_logmel(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """
    Args:
        waveform : 1-D or 2-D tensor (mono assumed if 2-D, single channel)
        sr       : source sample rate
    Returns:
        spec     : (1, 49, 40) float32 tensor in approximately [0, 1]
    """
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = T.Resample(sr, SAMPLE_RATE)(waveform)

    target_len = SAMPLE_RATE
    if waveform.shape[-1] < target_len:
        waveform = F.pad(waveform, (0, target_len - waveform.shape[-1]))
    else:
        waveform = waveform[..., :target_len]

    waveform = waveform.to(device)
    mel = _mel_transform(waveform)        # (1, 40, 49)
    mel = _amplitude_to_db(mel)
    mel = mel.squeeze(0).T                # (49, 40)
    mel = (mel + 80.0) / 80.0
    return mel.unsqueeze(0).cpu()         # (1, 49, 40)


# ── Sanity check + visualization ───────────────────────────────────────────────
test_wav = torch.randn(SAMPLE_RATE) * 0.05
spec     = waveform_to_logmel(test_wav, SAMPLE_RATE)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(spec.squeeze().numpy().T, aspect='auto', origin='lower', cmap='magma')
axes[0].set_xlabel('Time frame (×20 ms)'); axes[0].set_ylabel('Mel bin')
axes[0].set_title(f'Log-mel  shape={tuple(spec.shape)}')

aug = spec_augment(spec).squeeze().numpy().T
axes[1].imshow(aug, aspect='auto', origin='lower', cmap='magma')
axes[1].set_xlabel('Time frame (×20 ms)'); axes[1].set_ylabel('Mel bin')
axes[1].set_title('After SpecAugment (training only)')
plt.tight_layout(); plt.show()
assert spec.shape == (1, 49, 40), spec.shape
print('Feature extraction OK.')

---
## Section 4 — Embedding dataset

`fetch_kws_data.py` has already decoded every MSWC clip and saved 49×40 log-mel features to `feats/<kind>/<lang>/<word>.npy` on Drive. `stream_and_cache` below detects the pre-built files and skips HuggingFace entirely.

The cached arrays are float16 to halve disk I/O.

We also load a **held-out keyword bank** for Section 8, plus generate a **background-noise pool** synthetically.

**Balanced sampling**: MSWC is uneven across languages — English keywords may have 400 clips while Kinyarwanda keywords have 40. A naive shuffle would give English 10× the gradient signal. We use `WeightedRandomSampler` (Mazumder et al. recipe) to assign each sample weight `1 / class_size`, so every class contributes equally to every epoch regardless of clip count.

**Train / val split**: random 90/10 *within* each (lang, word) class. Balancing is applied only to the training split; the val split uses sequential order so per-language accuracy is reported on the true class distribution.

In [ ]:
# ── Load pre-computed features from Drive cache ─────────────────────────────
# fetch_kws_data.py has decoded every (lang, word) into a float16 .npy at
#   FEATS_DIR/{kind}/{lang}/{word}.npy   shape (N, 1, 49, 40)
# where N <= min(SAMPLES_PER_CLASS, clips available in metadata).
# We load each into memory; missing files (rare low-resource words) are
# reported but do not abort.

def _cache_path(lang: str, word: str, kind: str) -> Path:
    safe = word.replace('/', '_')
    return FEATS_DIR / kind / lang / f'{safe}.npy'


def load_cached(lang: str, words, kind: str) -> dict:
    """Load all available {word: ndarray} for a (lang, kind) split.
    Skips and warns about words with no cached .npy."""
    out, missing = {}, []
    for w in words:
        p = _cache_path(lang, w, kind)
        if p.exists():
            out[w] = np.load(p)
        else:
            missing.append(w)
    if missing:
        print(f'  [{lang}/{kind}] {len(missing)} words missing from cache: '
              f'{missing[:5]}{"..." if len(missing) > 5 else ""}')
    return out


train_cache, heldout_cache = {}, {}
for lang in LANGUAGES:
    inv = inventory[lang]
    train_cache[lang]   = load_cached(lang, inv['training'], 'train')
    heldout_cache[lang] = load_cached(lang, inv['heldout'],  'heldout')
    n_train = sum(arr.shape[0] for arr in train_cache[lang].values())
    n_held  = sum(arr.shape[0] for arr in heldout_cache[lang].values())
    print(f'  [{lang}] {len(train_cache[lang])}/{len(inv["training"])} train words '
          f'({n_train} samples), '
          f'{len(heldout_cache[lang])}/{len(inv["heldout"])} heldout words '
          f'({n_held} samples)')

total_train = sum(arr.shape[0] for d in train_cache.values()  for arr in d.values())
total_held  = sum(arr.shape[0] for d in heldout_cache.values() for arr in d.values())
print(f'Total training samples loaded: {total_train:,}')
print(f'Total training samples loaded: {total_train:,}')
print(f'Total held-out samples loaded: {total_held:,}')

if total_train == 0:
    raise RuntimeError(
        f'No cached features found under {FEATS_DIR}. '
        f'Run fetch_kws_data.py first (see Section 2).'
    )


In [ ]:
# ── Synthetic background-noise pool ───────────────────────────────────────────
# 10% of training samples are "noise" — this gives the embedding a
# language-agnostic "no keyword present" class. We blend three noise types:
#   (a) pure silence (zeros)
#   (b) low-amplitude white noise
#   (c) low-amplitude pink-ish noise (1/f via cumulative sum)
# Cheap, deterministic, no extra dependencies.

def make_noise_pool(n: int) -> np.ndarray:
    rng = np.random.RandomState(SEED)
    out = []
    for _ in range(n):
        kind = rng.choice(['silence', 'white', 'pink'])
        if kind == 'silence':
            w = np.zeros(SAMPLE_RATE, dtype=np.float32)
        elif kind == 'white':
            w = rng.randn(SAMPLE_RATE).astype(np.float32) * 0.005
        else:
            w = np.cumsum(rng.randn(SAMPLE_RATE)).astype(np.float32)
            w = (w - w.mean()) / (np.std(w) + 1e-9) * 0.005
        spec = waveform_to_logmel(torch.from_numpy(w), SAMPLE_RATE)
        out.append(spec.numpy().astype(np.float16))
    return np.stack(out, axis=0)


# Build noise pool sized to ~NOISE_FRACTION of the eventual training set
n_noise = int(NOISE_FRACTION * total_train / (1 - NOISE_FRACTION))
NOISE_CACHE_PATH = CACHE_DIR / 'noise_pool.npy'
if NOISE_CACHE_PATH.exists():
    noise_pool = np.load(NOISE_CACHE_PATH)
    print(f'Loaded {noise_pool.shape[0]} cached noise samples.')
else:
    print(f'Generating {n_noise} noise samples ...')
    noise_pool = make_noise_pool(n_noise)
    np.save(NOISE_CACHE_PATH, noise_pool)
print(f'Noise pool: {noise_pool.shape}  ({noise_pool.dtype})')

In [ ]:
# Build class label space + flat (spec, label) index
# Noise is NOT a class during embedding training - synthetic silence/white/pink
# is trivially separable from speech, contributes ~0 gradient once learned, and
# the balanced sampler would still allocate 1/N capacity to it.  The noise_pool
# is kept for Section 8 head training, where it serves as the 'background' class.
class_to_idx = {}
for lang in LANGUAGES:
    for word in inventory[lang]['training']:
        if word in train_cache.get(lang, {}):
            class_to_idx[(lang, word)] = len(class_to_idx)
idx_to_class = {v: k for k, v in class_to_idx.items()}
N_CLASSES = len(class_to_idx)

print(f'Total embedding classes: {N_CLASSES}  (noise excluded)')

lang_to_idx = {l: i for i, l in enumerate(LANGUAGES)}

train_index = []
for lang, by_word in train_cache.items():
    li = lang_to_idx[lang]
    for word, arr in by_word.items():
        cls = class_to_idx.get((lang, word))
        if cls is None:
            continue
        for j in range(arr.shape[0]):
            train_index.append((li, word, j, cls))

random.Random(SEED).shuffle(train_index)
print(f'Total flat training examples: {len(train_index):,}')

split = int(0.9 * len(train_index))
train_records = train_index[:split]
val_records   = train_index[split:]
print(f'Train: {len(train_records):,}   Val: {len(val_records):,}')

# Natural distribution training (no balanced sampler in v3).
# v2 used WeightedRandomSampler with replacement=True, which made the 1.55M
# v3 backbone memorise minority-class clips (each rw clip seen 5-10x/epoch).
# Cross-lingual transfer + higher per-class sample count (2000) mean we no
# longer need explicit balancing; rare classes ride on shared phonology.
class_counts = Counter(cls for _, _, _, cls in train_records)
print('\nClass size range (train split):')
sizes = sorted(class_counts.values())
print(f'  min={sizes[0]}  median={sizes[len(sizes)//2]}  max={sizes[-1]}')
print(f'  Imbalance ratio max/min: {sizes[-1]/max(sizes[0],1):.1f}x '
      f'(unbalanced; cross-lingual transfer covers rare classes)')


class EmbeddingDataset(Dataset):
    def __init__(self, records, augment: bool = False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        li, word, j, cls = self.records[idx]
        spec = train_cache[LANGUAGES[li]][word][j]
        spec = torch.from_numpy(spec.astype(np.float32))
        if self.augment:
            spec = spec_augment(spec)
        return spec, int(cls), int(li)


train_ds = EmbeddingDataset(train_records, augment=True)
val_ds   = EmbeddingDataset(val_records,   augment=False)

# DataLoader workers: on Colab (Linux + fork + copy-on-write) we can use
# multi-process loading at no extra RAM cost. On Windows local runs, spawn
# would deep-copy train_cache into each worker -> stay single-process.
_DL_WORKERS = 2 if _in_colab else 0
_DL_PERSIST = _in_colab and _DL_WORKERS > 0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=_DL_WORKERS,
                          persistent_workers=_DL_PERSIST,
                          pin_memory=(device.type == 'cuda'),
                          drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=_DL_WORKERS,
                          persistent_workers=_DL_PERSIST,
                          pin_memory=(device.type == 'cuda'))
print(f'Batches - train: {len(train_loader)}   val: {len(val_loader)}   '
      f'(num_workers={_DL_WORKERS})')


---
## Section 5 — DS-CNN backbone (256-dim multilingual embedding)

### Depthwise-separable convolution refresher

A standard Conv2d with kernel $k{\times}k$, $C_{in}$ input, $C_{out}$ output channels costs $O(k^2 C_{in} C_{out})$ per spatial location. A **depthwise separable block** factors this into:

- **Depthwise**: $k{\times}k$ filter applied independently per channel → $O(k^2 C_{in})$
- **Pointwise**: $1{\times}1$ conv to mix channels → $O(C_{in} C_{out})$

Total: $O(k^2 C_{in} + C_{in} C_{out})$ — roughly **8-9× cheaper** than standard conv for $k=3, C=64$. This is why DS-CNN became the standard TinyML KWS backbone (Zhang et al. 2017, "Hello Edge").

### Architecture

```
Input: (B, 1, 49, 40)

Conv2d(1→48, 3×3)        + BN + ReLU
DSBlock(48→64)
DSBlock(64→64)           MaxPool(2×2)
DSBlock(64→128, stride=2)
DSBlock(128→128)
DSBlock(128→192)
DSBlock(192→256)         GlobalAvgPool
                         ──▶  (B, 256)   ← THE EMBEDDING
                         Linear(256 → N_CLASSES)   ← training head (discarded at deploy)
```

The **penultimate 256-dim vector is the deployable artifact**. The final softmax exists only to give the embedding a training signal; we delete it at deployment and replace it with the tiny 5-shot heads from Section 8.

### Why these dimensions
- 48 → 256 channel ramp keeps the param count around 400-500K
- 256-dim embedding: Lei et al. 2023 swept this hyperparameter and found 256-512 is the sweet spot for medium backbones; 1024 actually hurt
- Two stride-2 reductions: spatial map goes 49×40 → 24×20 → 12×10 before GAP — enough resolution to localize phoneme structure

In [ ]:
class DSBlock(nn.Module):
    """Depthwise-separable conv block: DW(3x3)-BN-ReLU -> PW(1x1)-BN-ReLU."""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1,
                             groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x


class MultilingualEmbedding(nn.Module):
    """
    DS-CNN backbone -> 256-dim embedding -> N-way softmax (training only).
    v3: widened to ~1.55M params (5x v2). Targets ESP32-S3 + PSRAM deployment
    at ~300ms latency; sits in Mazumder-Small accuracy class.
    """

    def __init__(self, n_classes: int, embed_dim: int = EMBEDDING_DIM):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 96, 3, padding=1, bias=False),          # v2: 48
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            DSBlock(96, 192),                                    # v2: 48->64
            DSBlock(192, 192),
            nn.MaxPool2d(2),
            DSBlock(192, 320, stride=2),
            DSBlock(320, 320),
            DSBlock(320, 512),
            DSBlock(512, 640),
            DSBlock(640, 768),                                   # v2 final: 256
            nn.AdaptiveAvgPool2d(1),
        )
        self.project    = nn.Linear(768, embed_dim, bias=False)  # v2: 256
        self.embed_bn   = nn.BatchNorm1d(embed_dim)
        self.dropout    = nn.Dropout(0.2)                        # v3 regularizer
        self.classifier = nn.Linear(embed_dim, n_classes)

    def embed(self, x):
        h = self.backbone(x).flatten(1)         # (B, 768)
        h = self.project(h)                     # (B, embed_dim)
        h = self.embed_bn(h)
        return h

    def forward(self, x):
        emb = self.embed(x)
        logits = self.classifier(self.dropout(emb))
        return logits, emb

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = MultilingualEmbedding(n_classes=N_CLASSES, embed_dim=EMBEDDING_DIM).to(device)
print(f'Embedding model params : {model.count_params():,}')
print(f'Backbone + project only: {sum(p.numel() for n,p in model.named_parameters() if not n.startswith("classifier")):,}')

# Shape sanity check
dummy = torch.randn(4, 1, 49, 40, device=device)
logits, emb = model(dummy)
print(f'Logits shape: {tuple(logits.shape)}  (expect [4, {N_CLASSES}])')
print(f'Embed  shape: {tuple(emb.shape)}     (expect [4, {EMBEDDING_DIM}])')


---
## Section 6 — Embedding training

**Loss**: cross-entropy with label smoothing (0.1) over the N classes. Label smoothing pushes the model away from over-confident outputs, which empirically yields better embeddings for downstream few-shot transfer (Müller et al., NeurIPS 2019 — "When does label smoothing help?").

**Optimizer**: AdamW + cosine LR with warm-up. Standard recipe; no bespoke tuning.

**Evaluation during training**: top-1 accuracy overall + **per-language top-1 accuracy** (Mazumder Table 1 reproduction). The per-language breakdown surfaces low-resource issues early — if Persian or Kinyarwanda accuracy is dramatically below the others, we can decide whether to drop them before sinking time into 5-shot eval.

**Checkpointing**: save the best-val-accuracy state to `CKPT_PATH`. Section 7-8 load this checkpoint.

In [ ]:
@torch.no_grad()
def evaluate_embedding(loader, model) -> dict:
    """Top-1 accuracy overall + per language. Uses AMP autocast and on-GPU
    accumulation to avoid per-batch CPU syncs."""
    model.eval()
    use_amp = (device.type == 'cuda')

    correct_total = torch.zeros((), device=device, dtype=torch.long)
    total_total   = torch.zeros((), device=device, dtype=torch.long)
    # Per-language buckets on GPU (size = len(LANGUAGES))
    n_langs = len(LANGUAGES)
    correct_per_lang = torch.zeros(n_langs, device=device, dtype=torch.long)
    total_per_lang   = torch.zeros(n_langs, device=device, dtype=torch.long)

    for specs, labels, lang_idx in loader:
        specs    = specs.to(device, non_blocking=True)
        labels   = labels.to(device, non_blocking=True)
        lang_idx = lang_idx.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits, _ = model(specs)
        right = (logits.argmax(1) == labels)
        correct_total += right.sum()
        total_total   += labels.size(0)
        # scatter_add: per-lang correct/total without Python loop or .item()
        correct_per_lang.scatter_add_(0, lang_idx, right.long())
        total_per_lang.scatter_add_(0, lang_idx, torch.ones_like(lang_idx))

    overall = (correct_total.float() / total_total.clamp(min=1).float()).item()
    per_lang = {}
    cpl = correct_per_lang.cpu().tolist()
    tpl = total_per_lang.cpu().tolist()
    for li in range(n_langs):
        if tpl[li] > 0:
            per_lang[li] = cpl[li] / tpl[li]
    return {'overall': overall, 'per_lang': per_lang}


def train_embedding(model, train_loader, val_loader, epochs=EPOCHS,
                    lr=LR, weight_decay=WEIGHT_DECAY,
                    label_smooth=LABEL_SMOOTH, ckpt_path=CKPT_PATH) -> dict:
    """AMP-enabled training loop with GPU-resident metric accumulation.
    Eliminates per-batch CPU syncs (.item() calls) and uses fp16 matmul on
    tensor-core GPUs for 1.6-2x speedup. Behaviour and saved checkpoint
    format are identical to the pre-AMP version."""
    use_amp   = (device.type == 'cuda')
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader), pct_start=0.1,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
    history   = {'loss': [], 'train_acc': [], 'val_acc': [], 'val_per_lang': []}
    best_val  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss_t = torch.zeros((), device=device)
        correct_t    = torch.zeros((), device=device, dtype=torch.long)
        total_t      = torch.zeros((), device=device, dtype=torch.long)
        t0 = time.time()

        for specs, labels, _ in train_loader:
            specs  = specs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits, _ = model(specs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            with torch.no_grad():
                epoch_loss_t += loss.detach()
                correct_t    += (logits.argmax(1) == labels).sum()
                total_t      += labels.size(0)

        train_loss = (epoch_loss_t / len(train_loader)).item()
        train_acc  = (correct_t.float() / total_t.clamp(min=1).float()).item()
        val_stats  = evaluate_embedding(val_loader, model)

        history['loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_stats['overall'])
        history['val_per_lang'].append(val_stats['per_lang'])

        if val_stats['overall'] > best_val:
            best_val = val_stats['overall']
            torch.save({
                'state_dict': model.state_dict(),
                'class_to_idx': {f'{k[0]}|{k[1]}': v for k, v in class_to_idx.items()},
                'n_classes': N_CLASSES,
                'embed_dim': EMBEDDING_DIM,
                'epoch': epoch,
                'val_acc': best_val,
            }, ckpt_path)

        dt = time.time() - t0
        print(f'Ep {epoch:3d}/{epochs} | loss {train_loss:.3f} | '
              f'train {train_acc:.3f} | val {val_stats["overall"]:.3f} | '
              f'best {best_val:.3f} | {dt:.1f}s')

    print(f'\nBest val accuracy: {best_val:.3f}  (checkpoint: {ckpt_path})')
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['state_dict'])
    return history


HISTORY_PATH = CACHE_DIR / 'training_history.json'


def _save_history(h, path):
    serial = {
        'loss':         [float(x) for x in h['loss']],
        'train_acc':    [float(x) for x in h['train_acc']],
        'val_acc':      [float(x) for x in h['val_acc']],
        'val_per_lang': [{str(k): float(v) for k, v in d.items()}
                         for d in h['val_per_lang']],
    }
    path.write_text(json.dumps(serial, indent=2), encoding='utf-8')


def _load_history(path):
    raw = json.loads(path.read_text(encoding='utf-8'))
    raw['val_per_lang'] = [{int(k): float(v) for k, v in d.items()}
                           for d in raw['val_per_lang']]
    return raw


_can_load_train = CKPT_PATH.exists() and HISTORY_PATH.exists()
if TRAINING_RUN_MODE == 'cached' and not _can_load_train:
    raise FileNotFoundError(
        f'TRAINING_RUN_MODE="cached" requires both {CKPT_PATH} and {HISTORY_PATH}.'
    )

if TRAINING_RUN_MODE in ('auto', 'cached') and _can_load_train:
    state = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(state['state_dict'])
    history = _load_history(HISTORY_PATH)
    print(f'Loaded cached training: epoch={state.get("epoch", "?")}, '
          f'val_acc={state.get("val_acc", float("nan")):.3f}  '
          f'({len(history["loss"])} epochs of history)')
else:
    history = train_embedding(model, train_loader, val_loader)
    _save_history(history, HISTORY_PATH)
    print(f'Saved training history -> {HISTORY_PATH}')

In [ ]:
# ── Training curves + per-language final accuracy ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
eps = range(1, len(history['loss']) + 1)

axes[0].plot(eps, history['loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(eps, history['train_acc'], label='Train')
axes[1].plot(eps, history['val_acc'],   label='Val')
axes[1].set_title('Top-1 Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Final per-language accuracy (Mazumder Table 1 reproduction)
final_per_lang = history['val_per_lang'][-1]
ordered = [(LANGUAGES[li], acc) for li, acc in sorted(final_per_lang.items())
           if li >= 0]   # skip noise sentinel
langs, accs = zip(*ordered) if ordered else ([], [])
axes[2].barh(list(langs), list(accs), color=sns.color_palette('viridis', len(langs)))
axes[2].set_xlim(0, 1); axes[2].set_xlabel('Top-1 val accuracy')
axes[2].set_title('Per-language accuracy (final epoch)')
for i, a in enumerate(accs):
    axes[2].text(a + 0.01, i, f'{a:.2f}', va='center')

plt.tight_layout(); plt.savefig(CACHE_DIR / 'training_curves.png', dpi=150); plt.show()
print('\nFinal per-language accuracy:')
for lang_code, acc in ordered:
    print(f'  {lang_code:4s} {LANG_NAMES[lang_code]:12s}  {acc:.3f}')

---
## Section 7 - Embedding diagnostics

Two complementary views, both run on the loaded checkpoint (no retraining):

1. **Per-word t-SNE on held-out keywords**: this is the diagnostic that maps onto our actual
   downstream task (5-shot keyword detection). For each of 8 held-out words we project 30
   clips into 2-D and color by word. **If the embedding works for 5-shot, same-word clips
   should cluster** regardless of language. We also report the silhouette score, which
   numerically summarizes how separated those word-clusters are.

2. **Per-language t-SNE on training words** (kept for completeness): random clips per
   language colored by language. **Diffuse colors here are GOOD** - they mean the embedding
   learned phonological content rather than language identity. Clean language clusters
   would actually indicate the model is solving the wrong problem (language ID, not KWS),
   which would hurt cross-lingual transfer.

Cross-entropy training tends to produce embeddings with good *local* consistency
(same-word clips near each other) but messy *global* geometry. The per-word plot probes
the former (matters for 5-shot); the per-language plot probes the latter (matters less).


In [ ]:
# Defensive checkpoint load - lets this cell run standalone after a fresh kernel
if CKPT_PATH.exists():
    state = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(state['state_dict'])
    print(f'Loaded checkpoint: epoch={state.get("epoch", "?")}, '
          f'val_acc={state.get("val_acc", "?"):.3f}')

from sklearn.metrics import silhouette_score


@torch.no_grad()
def embed_array(arr: np.ndarray) -> np.ndarray:
    """Run the model.embed() over a (N,1,49,40) float16 array, return (N, EMBEDDING_DIM)."""
    model.eval()
    x = torch.from_numpy(arr.astype(np.float32)).to(device)
    return model.embed(x).cpu().numpy()


# ----- (1) Per-word t-SNE on held-out keywords -----------------------------
# Pick one held-out word from each of 8 languages for phonological diversity
N_WORDS_PLOT  = 8
N_CLIPS_WORD  = 30

selected = []
for lang in LANGUAGES:
    if len(selected) >= N_WORDS_PLOT:
        break
    for word in inventory[lang]['heldout']:
        arr = heldout_cache.get(lang, {}).get(word)
        if arr is not None and arr.shape[0] >= N_CLIPS_WORD:
            selected.append((lang, word, arr[:N_CLIPS_WORD]))
            break

feats_w, labels_w = [], []
for lang, word, arr in selected:
    embs = embed_array(arr)
    feats_w.append(embs)
    labels_w.extend([f'{lang}/{word}'] * embs.shape[0])
feats_w   = np.vstack(feats_w)
labels_w  = np.array(labels_w)
uniq_w    = list(dict.fromkeys(labels_w.tolist()))
label_idx = np.array([uniq_w.index(L) for L in labels_w])

print(f'Per-word pool: {feats_w.shape} over {len(uniq_w)} held-out words')

tsne_w = TSNE(n_components=2, perplexity=15, init='pca',
              learning_rate='auto', random_state=SEED)
xy_w = tsne_w.fit_transform(feats_w)

sil_word = silhouette_score(feats_w, label_idx)
print(f'Silhouette score (per-word, in 256-D embedding space): {sil_word:+.3f}')
print('  > 0  : same-word clips are closer to each other than to other words (good)')
print('  ~ 0  : no separation (bad)')
print('  < 0  : same-word clips are FARTHER apart than across-word (very bad)')

# Intra/inter mean distance ratio (cheaper interpretation)
from sklearn.metrics import pairwise_distances
D = pairwise_distances(feats_w, metric='cosine')
intra = D[label_idx[:, None] == label_idx[None, :]].mean()
inter = D[label_idx[:, None] != label_idx[None, :]].mean()
print(f'Mean cosine distance:  intra-word = {intra:.3f}   inter-word = {inter:.3f}   '
      f'ratio = {intra/inter:.3f}  (< 1 = useful embedding)')


# ----- (2) Per-language t-SNE on training words (kept) ---------------------
pools = defaultdict(list)
for li, word, j, cls in val_records:
    if li < 0:
        continue
    pools[li].append((word, j))

feats_l, langs_l = [], []
for li, items in pools.items():
    random.Random(SEED + li).shuffle(items)
    for word, j in items[:30]:
        spec = train_cache[LANGUAGES[li]][word][j:j+1]
        emb  = embed_array(spec)[0]
        feats_l.append(emb)
        langs_l.append(li)
feats_l = np.stack(feats_l)
langs_l = np.array(langs_l)

tsne_l = TSNE(n_components=2, perplexity=30, init='pca',
              learning_rate='auto', random_state=SEED)
xy_l = tsne_l.fit_transform(feats_l)

sil_lang = silhouette_score(feats_l, langs_l)
print(f'\nSilhouette score (per-language): {sil_lang:+.3f}  '
      f'(near 0 is GOOD - embedding is language-agnostic)')


# ----- Plot both side by side ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (a) per-word
palette_w = sns.color_palette('tab10', len(uniq_w))
for k, label in enumerate(uniq_w):
    m = labels_w == label
    axes[0].scatter(xy_w[m, 0], xy_w[m, 1], s=35, alpha=0.75,
                    color=palette_w[k], label=label)
axes[0].legend(loc='best', fontsize=9, framealpha=0.9)
axes[0].set_title(f'Per-word t-SNE on held-out keywords\n'
                  f'silhouette = {sil_word:+.3f}   intra/inter = {intra/inter:.3f}')
axes[0].set_xlabel('t-SNE dim 1'); axes[0].set_ylabel('t-SNE dim 2')

# (b) per-language
palette_l = sns.color_palette('tab10', len(LANGUAGES))
for li in range(len(LANGUAGES)):
    m = langs_l == li
    if m.any():
        axes[1].scatter(xy_l[m, 0], xy_l[m, 1], s=20, alpha=0.65,
                        color=palette_l[li],
                        label=f'{LANGUAGES[li]} ({LANG_NAMES[LANGUAGES[li]]})')
axes[1].legend(loc='best', fontsize=9, framealpha=0.9)
axes[1].set_title(f'Per-language t-SNE on training clips\n'
                  f'silhouette = {sil_lang:+.3f}   (near 0 is desired)')
axes[1].set_xlabel('t-SNE dim 1'); axes[1].set_ylabel('t-SNE dim 2')

plt.tight_layout(); plt.savefig(CACHE_DIR / 'tsne_diagnostics.png', dpi=150); plt.show()


---
## Section 8 — 5-shot head training & evaluation (headline experiment)

### Setup

For each `(language, held-out target word)` pair:

1. **Freeze** the embedding (no gradient flows back).
2. **Compute embeddings** for: 5 target samples + a balanced non-target pool drawn from MSWC clips of other words.
3. **Train a tiny 3-class softmax head** `Linear(256 → 3)` (target / unknown / background) for ~40 iterations on:
    - 5 target embeddings (positive class)
    - `N_UNKNOWN_PER_HEAD=128` non-target embeddings (unknown class)
    - `N_NOISE_PER_HEAD=25` noise embeddings (background class)
4. **Evaluate** on the remaining target samples (positive examples) and a fresh non-target pool (negative examples). Report F1 at threshold 0.8 (Mazumder convention).

### Why this experimental design

- **Out-of-embedding evaluation**: the held-out target keywords were not in the 450-class training set. So we're testing genuine generalization, not memorization.
- **5 samples** is the few-shot regime — what a real product flow looks like (user records 5 examples of "Hey, Toaster!" and the device starts listening for it).
- **Threshold = 0.8** lets us report F1 at a useful operating point; ROC-AUC is reported too for shape information.

### Expected result

Mazumder et al. report mean F1 ≈ 0.75 with their 11M-param EfficientNet embedding. Our DS-CNN is 20× smaller. **Anything in the 0.55-0.70 range is a strong outcome for our parameter budget**; matching their 0.75 would be a genuinely surprising result.

In [ ]:
# ── Helpers for 5-shot head training ──────────────────────────────────────────
TARGET_LBL, UNKNOWN_LBL, BG_LBL = 0, 1, 2

@torch.no_grad()
def embed_batch(specs: np.ndarray, model) -> np.ndarray:
    """specs: (N, 1, 49, 40) float16 -> embeddings (N, D) float32."""
    x = torch.from_numpy(specs.astype(np.float32)).to(device)
    return model.embed(x).cpu().numpy()


def build_unknown_pool(target_lang: str, target_word: str,
                       n_samples: int, model) -> np.ndarray:
    """
    Build a non-target pool by drawing from MSWC heldout words in OTHER languages
    + heldout words in the same language other than target_word.
    """
    pool_specs = []
    for lang in LANGUAGES:
        for word, arr in heldout_cache.get(lang, {}).items():
            if lang == target_lang and word == target_word:
                continue
            # take a few per word
            k = min(4, arr.shape[0])
            pool_specs.append(arr[:k])
    pool = np.concatenate(pool_specs, axis=0)
    # sample exactly n_samples
    rng = np.random.RandomState(SEED + hash(target_word) % 10_000)
    idx = rng.choice(pool.shape[0], size=min(n_samples, pool.shape[0]), replace=False)
    return embed_batch(pool[idx], model)


def train_fewshot_head(target_embs: np.ndarray, unknown_embs: np.ndarray,
                       bg_embs: np.ndarray, embed_dim: int = EMBEDDING_DIM,
                       epochs: int = HEAD_EPOCHS, lr: float = HEAD_LR) -> nn.Module:
    """
    Tiny 3-class linear head trained on top of frozen embeddings.
    Returns the trained head (on `device`).
    """
    X = np.concatenate([target_embs, unknown_embs, bg_embs], axis=0)
    y = np.concatenate([
        np.full(target_embs.shape[0],  TARGET_LBL,  dtype=np.int64),
        np.full(unknown_embs.shape[0], UNKNOWN_LBL, dtype=np.int64),
        np.full(bg_embs.shape[0],      BG_LBL,      dtype=np.int64),
    ])
    # Weight targets up to compensate for the imbalance (5 vs 128 vs 25)
    class_weight = torch.tensor([
        (X.shape[0] / 3.0) / max(target_embs.shape[0], 1),
        (X.shape[0] / 3.0) / max(unknown_embs.shape[0], 1),
        (X.shape[0] / 3.0) / max(bg_embs.shape[0], 1),
    ], dtype=torch.float32, device=device)

    head = nn.Linear(embed_dim, 3).to(device)
    opt  = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weight)
    Xt = torch.from_numpy(X).to(device)
    yt = torch.from_numpy(y).to(device)
    for _ in range(epochs):
        opt.zero_grad()
        loss = crit(head(Xt), yt)
        loss.backward(); opt.step()
    return head


def eval_fewshot_head(head: nn.Module, target_test_embs: np.ndarray,
                      negative_test_embs: np.ndarray, threshold: float = F1_THRESHOLD) -> dict:
    """
    Returns dict with:
      - f1, precision, recall, roc_auc, tpr/fpr at fixed threshold (Mazumder convention 0.8)
      - f1_opt, thr_opt: F1 at the per-keyword optimal threshold (fairer for small models)
      - roc_curve: (fpr, tpr) arrays for plotting
    Positive = target class. Decision = (P[target] > threshold).
    """
    from sklearn.metrics import precision_recall_curve
    head.eval()
    with torch.no_grad():
        pos_logits = head(torch.from_numpy(target_test_embs).to(device))
        neg_logits = head(torch.from_numpy(negative_test_embs).to(device))
    pos_prob = F.softmax(pos_logits, dim=1)[:, TARGET_LBL].cpu().numpy()
    neg_prob = F.softmax(neg_logits, dim=1)[:, TARGET_LBL].cpu().numpy()

    y_true   = np.concatenate([np.ones_like(pos_prob), np.zeros_like(neg_prob)])
    y_score  = np.concatenate([pos_prob, neg_prob])
    y_pred   = (y_score >= threshold).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc     = auc(fpr, tpr)
    except Exception:
        fpr, tpr, roc_auc = np.array([0., 1.]), np.array([0., 1.]), float('nan')

    # Optimal-threshold F1 (sweeps all thresholds, picks argmax)
    try:
        precs, recs, thrs = precision_recall_curve(y_true, y_score)
        f1_curve = 2 * precs * recs / (precs + recs + 1e-9)
        best_idx = int(np.argmax(f1_curve))
        f1_opt   = float(f1_curve[best_idx])
        # thrs has length len(precs)-1; clip best_idx if it falls on the trailing point
        thr_opt  = float(thrs[min(best_idx, len(thrs)-1)]) if len(thrs) > 0 else 0.5
    except Exception:
        f1_opt, thr_opt = float('nan'), float('nan')

    return {'f1': f1, 'precision': prec, 'recall': rec, 'roc_auc': roc_auc,
            'tpr_at_thr': rec, 'fpr_at_thr': fp / max(fp + tn, 1),
            'f1_opt': f1_opt, 'thr_opt': thr_opt,
            'roc_curve': (fpr, tpr)}


print('5-shot helpers defined.')

In [ ]:
# ── Sweep across (lang, heldout-word) pairs (with run-mode gate) ─────────────
FEWSHOT_RESULTS_PATH = CACHE_DIR / 'fewshot_results.json'

_FEWSHOT_KEYS = ('f1', 'precision', 'recall', 'roc_auc', 'tpr_at_thr', 'fpr_at_thr',
                 'f1_opt', 'thr_opt', 'lang', 'word', 'n_test_pos')


def _save_fewshot(results, path):
    out = []
    for r in results:
        d = {k: r[k] for k in _FEWSHOT_KEYS if k in r}
        for k, v in d.items():
            if hasattr(v, 'item'):
                d[k] = v.item()
        fpr, tpr = r.get('roc_curve', (np.array([0., 1.]), np.array([0., 1.])))
        d['roc_curve'] = [list(map(float, fpr)), list(map(float, tpr))]
        out.append(d)
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding='utf-8')


def _load_fewshot(path):
    raw = json.loads(path.read_text(encoding='utf-8'))
    missing_roc = 0
    missing_opt = 0
    for r in raw:
        rc = r.get('roc_curve')
        if rc is None:
            r['roc_curve'] = (np.array([0., 1.]), np.array([0., 1.]))
            missing_roc += 1
        else:
            fpr, tpr = rc
            r['roc_curve'] = (np.asarray(fpr, dtype=float), np.asarray(tpr, dtype=float))
        for k in ('f1', 'precision', 'recall', 'roc_auc', 'tpr_at_thr',
                  'fpr_at_thr', 'f1_opt', 'thr_opt'):
            if k in r and r[k] is not None:
                try: r[k] = float(r[k])
                except (TypeError, ValueError): pass
            else:
                if k in ('f1_opt', 'thr_opt'):
                    missing_opt += 1
                r.setdefault(k, float('nan'))
    if missing_roc:
        print(f'Note: {missing_roc}/{len(raw)} cached entries lack roc_curve '
              f'(old-format cache). ROC plot will show diagonals for those.')
    if missing_opt:
        print(f'Note: {missing_opt//2} cached entries lack f1_opt/thr_opt; re-run with '
              f'FEWSHOT_RUN_MODE='fresh' to populate them.')
    return raw


_can_load_fs = FEWSHOT_RESULTS_PATH.exists()
if FEWSHOT_RUN_MODE == 'cached' and not _can_load_fs:
    raise FileNotFoundError(
        f'FEWSHOT_RUN_MODE="cached" but no results at {FEWSHOT_RESULTS_PATH}.'
    )

if FEWSHOT_RUN_MODE in ('auto', 'cached') and _can_load_fs:
    fewshot_results = _load_fewshot(FEWSHOT_RESULTS_PATH)
    print(f'Loaded {len(fewshot_results)} cached 5-shot results from {FEWSHOT_RESULTS_PATH}')
    # bg_embs_full is still needed for Section 8b even when sweep is cached
    if 'bg_embs_full' not in dir():
        bg_embs_full = embed_batch(noise_pool[:max(N_NOISE_PER_HEAD * 4, 100)], model)
else:
    fewshot_results = []
    n_pairs = 0
    for lang in LANGUAGES:
        for word in inventory[lang]['heldout']:
            if word not in heldout_cache.get(lang, {}):
                continue
            n_pairs += 1
    print(f'Sweeping {n_pairs} (lang, heldout-word) pairs...')

    bg_embs_full = embed_batch(noise_pool[:max(N_NOISE_PER_HEAD * 4, 100)], model)

    t_start = time.time()
    for lang in LANGUAGES:
        for word in inventory[lang]['heldout']:
            if word not in heldout_cache.get(lang, {}):
                continue
            arr = heldout_cache[lang][word]
            if arr.shape[0] < N_SHOT + 30:
                continue
            rng = np.random.RandomState(SEED + hash(f'{lang}|{word}') % 10_000)
            perm = rng.permutation(arr.shape[0])
            shot_idx = perm[:N_SHOT]
            test_idx = perm[N_SHOT:N_SHOT + 100]

            target_train_embs = embed_batch(arr[shot_idx], model)
            target_test_embs  = embed_batch(arr[test_idx], model)
            unknown_embs      = build_unknown_pool(lang, word, N_UNKNOWN_PER_HEAD, model)
            bg_idx = rng.choice(bg_embs_full.shape[0],
                                size=min(N_NOISE_PER_HEAD, bg_embs_full.shape[0]),
                                replace=False)
            bg_embs = bg_embs_full[bg_idx]

            head = train_fewshot_head(target_train_embs, unknown_embs, bg_embs)

            neg_pool_embs = build_unknown_pool(lang, word, 200, model)
            metrics = eval_fewshot_head(head, target_test_embs, neg_pool_embs)
            metrics['lang']  = lang
            metrics['word']  = word
            metrics['n_test_pos'] = int(target_test_embs.shape[0])
            fewshot_results.append(metrics)

    print(f'\n5-shot sweep done in {time.time()-t_start:.1f}s. '
          f'{len(fewshot_results)} keywords evaluated.')

    _save_fewshot(fewshot_results, FEWSHOT_RESULTS_PATH)
    print(f'Saved 5-shot results -> {FEWSHOT_RESULTS_PATH}')

# ── Summary table (always print, both fresh and cached) ─────────────────────
def _mean_nan(xs):
    xs = [x for x in xs if not math.isnan(x)]
    return float(np.mean(xs)) if xs else float('nan')

print(f'\n{"lang":6s} {"#kw":>4s} {"F1@0.8":>8s} {"F1@opt":>8s} '
      f'{"AUC":>7s} {"thr_opt":>9s}')
per_lang_summary = []
for lang in LANGUAGES:
    items = [r for r in fewshot_results if r['lang'] == lang]
    if not items:
        continue
    f1s     = [r['f1']      for r in items]
    f1opts  = [r['f1_opt']  for r in items]
    aucs    = [r['roc_auc'] for r in items]
    thropts = [r['thr_opt'] for r in items]
    per_lang_summary.append((lang, len(items), _mean_nan(f1s), _mean_nan(f1opts),
                             _mean_nan(aucs), _mean_nan(thropts)))
    print(f'{lang:6s} {len(items):4d} {_mean_nan(f1s):8.3f} {_mean_nan(f1opts):8.3f} '
          f'{_mean_nan(aucs):7.3f} {_mean_nan(thropts):9.3f}')

all_f1     = [r['f1']      for r in fewshot_results]
all_f1opt  = [r['f1_opt']  for r in fewshot_results]
all_aucs   = [r['roc_auc'] for r in fewshot_results]
print(f'\nOverall mean F1 @ {F1_THRESHOLD:.1f} (Mazumder convention): {_mean_nan(all_f1):.3f}  '
      f'(Mazumder 11M EfficientNet: 0.75)')
print(f'Overall mean F1 @ per-keyword optimal threshold: {_mean_nan(all_f1opt):.3f}  '
      f'(fairer for small models)')
print(f'Overall mean ROC-AUC                            : {_mean_nan(all_aucs):.3f}')


In [ ]:
# ── Visualize 5-shot results (Mazumder Fig. 2 style) ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (a) F1 distribution per language
df_rows = []
for r in fewshot_results:
    df_rows.append({'lang': r['lang'], 'F1': r['f1'], 'AUC': r['roc_auc']})
langs_present = sorted({r['lang'] for r in fewshot_results}, key=LANGUAGES.index)
palette = sns.color_palette('tab10', len(LANGUAGES))
lang_color = {LANGUAGES[i]: palette[i] for i in range(len(LANGUAGES))}

# seaborn >=0.13 requires DataFrame or Mapping (not list of dicts)
df_rows = {
    'lang': [row['lang'] for row in df_rows],
    'F1':   [row['F1'] for row in df_rows],
    'AUC':  [row['AUC'] for row in df_rows],
}

sns.boxplot(
    x='lang', y='F1', data=df_rows, order=langs_present,
    palette=[lang_color[l] for l in langs_present], ax=axes[0],
)
sns.stripplot(
    x='lang', y='F1', data=df_rows, order=langs_present,
    color='black', size=3, alpha=0.5, ax=axes[0],
)
axes[0].axhline(0.75, color='red', linestyle='--', alpha=0.6,
                label='Mazumder (11M EffNet)')
axes[0].set_ylim(0, 1.0); axes[0].set_title(f'5-shot F1 @ threshold {F1_THRESHOLD} per language')
axes[0].legend()

# (b) ROC curves overlay per language (mean over its keywords)
for lang in langs_present:
    items = [r for r in fewshot_results if r['lang'] == lang]
    # Common FPR grid
    grid = np.linspace(0, 1, 200)
    tprs = []
    for r in items:
        fpr, tpr = r['roc_curve']
        tprs.append(np.interp(grid, fpr, tpr))
    if not tprs:
        continue
    mean_tpr = np.mean(tprs, axis=0)
    axes[1].plot(grid, mean_tpr, color=lang_color[lang],
                 label=f'{lang} ({LANG_NAMES[lang]}) AUC={np.mean([r["roc_auc"] for r in items]):.2f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title('Mean ROC per language (5-shot)')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout(); plt.savefig(CACHE_DIR / 'fewshot_results.png', dpi=150); plt.show()

# Save results table for the report
results_path = CACHE_DIR / 'fewshot_results.json'
results_path.write_text(json.dumps(
    [{'lang': r['lang'], 'word': r['word'], 'f1': r['f1'],
      'roc_auc': r['roc_auc'], 'precision': r['precision'],
      'recall': r['recall']} for r in fewshot_results],
    indent=2, ensure_ascii=False,
), encoding='utf-8')
print(f'Results table saved: {results_path}')

---
## Section 8b — Optional: TTS-enrolled heads (Lin et al. reproduction)

**Skip this section if Colab time is tight — it's a supplement, not part of the headline.**

The Section 8 5-shot eval uses *real* MSWC clips as the 5 target samples. But in a real deployment, a user might want to teach the device a brand-new wake word that doesn't exist in any corpus (a product name, an invented phrase). The natural answer is **TTS-generated samples**.

Lin et al. (ICASSP 2020) showed that a **head model on top of a real-audio embedding can be trained entirely on synthetic speech** and reach 92.6% on a real-audio test set — within 2% of the same head trained on real speech. We reproduce that finding here with our small-backbone embedding: for each test keyword, we generate 5 TTS samples (using the existing `generate_tts.py` infrastructure) and train the head on those instead of MSWC samples. The held-out **test set remains real MSWC clips**.

Result we expect: 5-shot F1 from TTS-enrolled heads should be **within ~5% of real-enrolled F1** for languages with diverse TTS voices, validating the "5 TTS samples can replace 5 real samples" deployment claim.

The code below is wrapped in a `try/except` — if `edge_tts` isn't available, the section is skipped without affecting Section 9.

In [ ]:
# Optional: TTS-enrolled heads (Lin et al. 2020 reproduction).
# Requires `edge-tts`; install with: !pip install -q edge-tts nest_asyncio

# ---- Run-mode switch ----------------------------------------------------
# 'auto'   : load cached results if the JSON exists, otherwise run fresh
# 'fresh'  : always synthesize TTS and re-train heads (slow, ~5-10 min)
# 'cached' : only load from JSON; error if missing
# TTS_RUN_MODE is set in Section 1's config cell (defaults to RUN_MODE)
TTS_RESULTS_PATH = CACHE_DIR / 'tts_fewshot_results.json'

TTS_VOICES = {
    'en': ['en-US-AriaNeural', 'en-US-GuyNeural', 'en-GB-RyanNeural',
           'en-GB-SoniaNeural', 'en-AU-WilliamNeural'],
    'de': ['de-DE-KatjaNeural', 'de-DE-ConradNeural', 'de-AT-IngridNeural',
           'de-CH-LeniNeural', 'de-DE-AmalaNeural'],
    'fr': ['fr-FR-DeniseNeural', 'fr-FR-HenriNeural', 'fr-CA-SylvieNeural',
           'fr-CA-AntoineNeural', 'fr-CH-ArianeNeural'],
    'es': ['es-ES-ElviraNeural', 'es-ES-AlvaroNeural', 'es-MX-DaliaNeural',
           'es-MX-JorgeNeural', 'es-AR-ElenaNeural'],
    'fa': ['fa-IR-DilaraNeural', 'fa-IR-FaridNeural'],
    'ar': ['ar-EG-SalmaNeural', 'ar-EG-ShakirNeural', 'ar-SA-ZariyahNeural',
           'ar-SA-HamedNeural', 'ar-AE-FatimaNeural'],
}

_TTS_SUMMARY_KEYS = ('f1', 'precision', 'recall', 'roc_auc',
                     'tpr_at_thr', 'fpr_at_thr', 'f1_opt', 'thr_opt',
                     'lang', 'word', 'enrollment', 'n_tts_clips')


def _serialize_tts_results(results):
    """Drop the (fpr, tpr) arrays before saving; coerce numpy scalars to Python."""
    out = []
    for r in results:
        d = {k: r[k] for k in _TTS_SUMMARY_KEYS if k in r}
        for k, v in d.items():
            if hasattr(v, 'item'):
                d[k] = v.item()
        out.append(d)
    return out


def _synth_tts_embed(lang, word, model, n_shot=N_SHOT):
    """Synthesize up to n_shot TTS clips for (lang, word) and return their embeddings.
    If the language has fewer than n_shot voices, returns what's available (>=2).
    Returns None on hard failure."""
    import asyncio, edge_tts, tempfile
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        pass

    voices = TTS_VOICES.get(lang, [])
    if not voices:
        return None
    voices = voices[:n_shot]

    embs = []
    for voice in voices:
        outpath = None
        try:
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
                outpath = f.name
            async def _gen():
                comm = edge_tts.Communicate(text=word, voice=voice)
                await comm.save(outpath)
            asyncio.run(_gen())
            wav, sr = torchaudio.load(outpath)
            spec = waveform_to_logmel(wav.squeeze(0), sr)
            with torch.no_grad():
                emb = model.embed(spec.unsqueeze(0).to(device)).cpu().numpy().squeeze(0)
            embs.append(emb)
        except Exception as exc:
            print(f'      voice {voice}: {type(exc).__name__}: {exc}')
        finally:
            if outpath and os.path.exists(outpath):
                try: os.remove(outpath)
                except OSError: pass

    if len(embs) < 2:
        return None
    return np.stack(embs).astype(np.float32)


tts_results = []
_loaded_from_cache = False

# ---- Load cached results if requested ----------------------------------
if TTS_RUN_MODE in ('auto', 'cached') and TTS_RESULTS_PATH.exists():
    try:
        tts_results = json.loads(TTS_RESULTS_PATH.read_text(encoding='utf-8'))
        _loaded_from_cache = True
        print(f'Loaded {len(tts_results)} cached TTS results from {TTS_RESULTS_PATH}')
    except Exception as exc:
        print(f'Could not read {TTS_RESULTS_PATH}: {exc}')
        tts_results = []

if TTS_RUN_MODE == 'cached' and not _loaded_from_cache:
    raise FileNotFoundError(
        f'TTS_RUN_MODE="cached" but no results at {TTS_RESULTS_PATH}. '
        f'Run once with TTS_RUN_MODE="fresh" or "auto" to populate it.'
    )

# ---- Fresh synthesis sweep ---------------------------------------------
if not _loaded_from_cache:
    try:
        import edge_tts   # noqa: F401

        # Defensive bootstrap: every variable Section 8 produced that we need here.
        if 'bg_embs_full' not in dir():
            if 'noise_pool' not in dir():
                raise RuntimeError('noise_pool missing - re-run Section 4 (cell 13) first.')
            print('bg_embs_full missing; rebuilding from noise_pool ...')
            bg_embs_full = embed_batch(noise_pool[:max(N_NOISE_PER_HEAD * 4, 100)], model)

        comparison_pairs = []
        for lang in LANGUAGES:
            if lang not in TTS_VOICES:
                continue
            kws = [w for w in inventory[lang]['heldout']
                   if w in heldout_cache.get(lang, {})
                   and heldout_cache[lang][w].shape[0] >= N_SHOT + 30]
            comparison_pairs.extend((lang, w) for w in kws[:5])
        print(f'Running TTS-enrolled heads on {len(comparison_pairs)} keywords '
              f'across {len(TTS_VOICES)} TTS-supported languages.')

        for lang, word in comparison_pairs:
            try:
                tts_embs = _synth_tts_embed(lang, word, model)
                if tts_embs is None:
                    print(f'  SKIP {lang}/{word}: <2 TTS clips synthesized')
                    continue

                arr = heldout_cache[lang][word]
                rng = np.random.RandomState(SEED + hash(f'{lang}|{word}|tts') % 10_000)
                test_idx = rng.permutation(arr.shape[0])[:100]
                target_test_embs = embed_batch(arr[test_idx], model)
                unknown_embs     = build_unknown_pool(lang, word, N_UNKNOWN_PER_HEAD, model)
                bg_idx = rng.choice(bg_embs_full.shape[0],
                                    size=min(N_NOISE_PER_HEAD, bg_embs_full.shape[0]),
                                    replace=False)
                bg_embs = bg_embs_full[bg_idx]

                head = train_fewshot_head(tts_embs, unknown_embs, bg_embs)
                neg_pool_embs = build_unknown_pool(lang, word, 200, model)
                m = eval_fewshot_head(head, target_test_embs, neg_pool_embs)
                m.update({'lang': lang, 'word': word, 'enrollment': 'TTS',
                          'n_tts_clips': int(tts_embs.shape[0])})
                tts_results.append(m)
                print(f'  TTS  {lang}/{word:20s} '
                      f'F1={m["f1"]:.3f}  AUC={m["roc_auc"]:.3f}  '
                      f'(n_clips={tts_embs.shape[0]})')
            except Exception as exc:
                print(f'  ERR  {lang}/{word}: {type(exc).__name__}: {exc}')

        # Persist for next session
        if tts_results:
            TTS_RESULTS_PATH.write_text(
                json.dumps(_serialize_tts_results(tts_results), indent=2),
                encoding='utf-8',
            )
            print(f'\nSaved {len(tts_results)} TTS results -> {TTS_RESULTS_PATH}')

    except ImportError:
        print('edge-tts not installed.  Run: !pip install -q edge-tts nest_asyncio')
    except Exception as exc:
        print(f'Section 8b failed (non-fatal): {type(exc).__name__}: {exc}')


# ---- Summaries (work for both fresh and cached results) ----------------
if tts_results:
    print(f'\nTTS-enrolled summary{" (cached)" if _loaded_from_cache else ""}:')
    print(f'{"lang":6s} {"#kw":>4s} {"F1@0.8":>8s} {"F1@opt":>8s} {"AUC":>7s}')
    for lang in LANGUAGES:
        items = [r for r in tts_results if r['lang'] == lang]
        if not items: continue
        f1s    = [r.get('f1', float('nan'))      for r in items]
        f1opts = [r.get('f1_opt', float('nan'))  for r in items]
        aucs   = [r.get('roc_auc', float('nan')) for r in items]
        def _m(xs):
            xs = [x for x in xs if not math.isnan(x)]
            return float(np.mean(xs)) if xs else float('nan')
        print(f'{lang:6s} {len(items):4d} {_m(f1s):8.3f} {_m(f1opts):8.3f} {_m(aucs):7.3f}')
    def _M(key):
        xs = [r.get(key, float('nan')) for r in tts_results]
        xs = [x for x in xs if not math.isnan(x)]
        return float(np.mean(xs)) if xs else float('nan')
    print(f'\nOverall mean F1@0.8  (TTS) : {_M("f1"):.3f}')
    print(f'Overall mean F1@opt  (TTS) : {_M("f1_opt"):.3f}')
    print(f'Overall mean AUC     (TTS) : {_M("roc_auc"):.3f}')

    have_real_baseline = 'fewshot_results' in dir() and fewshot_results
    if have_real_baseline:
        comp_real     = {(r['lang'], r['word']): r['f1']     for r in fewshot_results}
        comp_real_opt = {(r['lang'], r['word']): r.get('f1_opt', float('nan'))
                         for r in fewshot_results}
        rows = []
        for r in tts_results:
            key = (r['lang'], r['word'])
            if key in comp_real:
                rows.append((r['lang'], r['word'],
                             comp_real[key], r['f1'],
                             comp_real_opt[key], r.get('f1_opt', float('nan'))))
        if rows:
            print(f'\n{"lang/word":30s} {"F1real@0.8":>11s} {"F1tts@0.8":>10s} '
                  f'{"F1real@opt":>11s} {"F1tts@opt":>10s} {"d@opt":>7s}')
            for lang, word, fr, ft, fro, fto in rows:
                d_opt = fto - fro if not (math.isnan(fto) or math.isnan(fro)) else float('nan')
                print(f'{lang}/{word:25s} {fr:11.3f} {ft:10.3f} {fro:11.3f} {fto:10.3f} '
                      f'{d_opt:+7.3f}' if not math.isnan(d_opt)
                      else f'{lang}/{word:25s} {fr:11.3f} {ft:10.3f} {fro:11.3f} {fto:10.3f} '
                           f'{"nan":>7s}')
            deltas_fixed = [ft - fr for _, _, fr, ft, _, _ in rows]
            deltas_opt   = [fto - fro for _, _, _, _, fro, fto in rows
                             if not (math.isnan(fto) or math.isnan(fro))]
            print(f'\nMean F1 delta (TTS - real) @ 0.8         : {np.mean(deltas_fixed):+.3f}')
            if deltas_opt:
                print(f'Mean F1 delta (TTS - real) @ opt thresh  : {np.mean(deltas_opt):+.3f}')
            print(f'(Lin et al. 2020 reference: TTS-enrolled within ~2pp of real-enrolled)')
    else:
        print('\nfewshot_results not in memory - real-vs-TTS comparison skipped. '
              'Re-run Section 8 to enable it.')
else:
    print('\nNo TTS results to summarise.')


---
## Section 8c - Novel-keyword deployment vignette

Section 8b measured TTS-enrollment **for words that exist in MSWC** - a methodological proxy. The actual deployment claim is broader: *a user can teach the device a wake word that wasn't in any training corpus*.

This section runs that demonstration for **6 novel keywords, one per language** (chosen to be distinctive content words not in any MSWC top-50 list). For each keyword we:

1. Synthesize 5 enrollment samples using diverse edge-tts voices for that language
2. Synthesize 1 held-out *positive* test sample (different voice from enrollment)
3. Pick 5 *negative* test samples from MSWC heldout clips (any language)
4. Train a head on the 5 enrollment samples + standard unknown/background pool
5. Score the positive and negative samples, report whether the head can distinguish them

**Caveats** (this is a qualitative vignette, not a benchmark):
- No real-microphone evaluation - all positive samples are TTS
- Voice coverage varies by language; ar has the most variants, fa has the fewest
- The TTS-vs-real domain gap measured in Section 8b applies here too

**Pass criteria:** held-out positive score > 0.5 AND mean negative score < 0.5.


In [ ]:
# Novel-keyword TTS demo. Self-contained: depends on `model`, `embed_batch`,
# `build_unknown_pool`, `train_fewshot_head`, `bg_embs_full`, `heldout_cache`.
# All produced earlier in the notebook.

NOVEL_KEYWORDS = {
    'en': 'Marvin',
    'de': 'Donner',
    'fr': 'Etoile',
    'es': 'Murcielago',
    'fa': 'آسمان',     # asman (sky)
    'ar': 'نجم',                  # najm (star)
}

NOVEL_RESULTS_PATH = CACHE_DIR / 'novel_keyword_results.json'


def _list_voices_for(lang_prefix, all_voices):
    """Filter a flat edge-tts voice list down to one language family."""
    return [v['ShortName'] for v in all_voices
            if v['ShortName'].lower().startswith(f'{lang_prefix}-')]


def _synth_word(text, voice):
    """Single TTS synthesis -> (1, 49, 40) float32 embedding via the loaded model."""
    import asyncio, edge_tts, tempfile
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        pass
    outpath = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            outpath = f.name
        async def _gen():
            comm = edge_tts.Communicate(text=text, voice=voice)
            await comm.save(outpath)
        asyncio.run(_gen())
        wav, sr = torchaudio.load(outpath)
        spec = waveform_to_logmel(wav.squeeze(0), sr)
        with torch.no_grad():
            emb = model.embed(spec.unsqueeze(0).to(device)).cpu().numpy().squeeze(0)
        return emb.astype(np.float32)
    except Exception as exc:
        print(f'    synth FAIL voice={voice}: {type(exc).__name__}: {exc}')
        return None
    finally:
        if outpath and os.path.exists(outpath):
            try: os.remove(outpath)
            except OSError: pass


novel_results = []
try:
    import asyncio, edge_tts
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        pass

    # Defensive bootstrap of bg_embs_full (same pattern as Section 8b)
    if 'bg_embs_full' not in dir():
        bg_embs_full = embed_batch(noise_pool[:max(N_NOISE_PER_HEAD * 4, 100)], model)

    print('Discovering edge-tts voices...')
    all_voices = asyncio.run(edge_tts.list_voices())
    voices_by_lang = {lang: _list_voices_for(lang, all_voices) for lang in NOVEL_KEYWORDS}
    for lang, vs in voices_by_lang.items():
        print(f'  {lang}: {len(vs)} voices available')

    # Build a single shared MSWC distractor pool: random heldout clips from all langs
    mswc_distractors = []
    rng = np.random.RandomState(SEED + 7777)
    for lang in LANGUAGES:
        for word, arr in heldout_cache.get(lang, {}).items():
            k = min(2, arr.shape[0])
            idx = rng.choice(arr.shape[0], size=k, replace=False)
            mswc_distractors.append(arr[idx])
    mswc_distractors = np.concatenate(mswc_distractors, axis=0)
    print(f'MSWC distractor pool: {mswc_distractors.shape[0]} clips')
    mswc_distractor_embs = embed_batch(mswc_distractors[:300], model)

    # ----- Per-keyword demo loop -----
    for lang, keyword in NOVEL_KEYWORDS.items():
        voices = voices_by_lang.get(lang, [])
        if len(voices) < 2:
            print(f'  SKIP {lang}/{keyword}: only {len(voices)} TTS voices')
            continue
        rng_kw = np.random.RandomState(SEED + abs(hash(keyword)) % 10_000)
        rng_kw.shuffle(voices)
        enroll_voices = voices[:5]
        holdout_voices = voices[5:6] if len(voices) >= 6 else voices[:1]  # reuse if scarce

        # Enrollment
        enroll_embs = []
        for v in enroll_voices:
            emb = _synth_word(keyword, v)
            if emb is not None:
                enroll_embs.append(emb)
        if len(enroll_embs) < 2:
            print(f'  SKIP {lang}/{keyword}: only {len(enroll_embs)} enrollment clips synthesized')
            continue
        enroll_embs = np.stack(enroll_embs).astype(np.float32)

        # Held-out positive
        holdout_emb = None
        for v in holdout_voices:
            holdout_emb = _synth_word(keyword, v)
            if holdout_emb is not None:
                break

        # Negative pool: other novel keywords (cross-language TTS) + MSWC distractors
        cross_embs = []
        for other_lang, other_kw in NOVEL_KEYWORDS.items():
            if other_kw == keyword:
                continue
            other_voices = voices_by_lang.get(other_lang, [])
            if not other_voices:
                continue
            emb = _synth_word(other_kw, other_voices[0])
            if emb is not None:
                cross_embs.append(emb)
        cross_embs = np.stack(cross_embs).astype(np.float32) if cross_embs else np.zeros((0, EMBEDDING_DIM), dtype=np.float32)

        # Unknown + background for head training
        # Use a dummy 'target_word' that won't collide with real MSWC heldout
        unknown_embs = build_unknown_pool(lang, '__novel__', N_UNKNOWN_PER_HEAD, model)
        bg_idx = rng_kw.choice(bg_embs_full.shape[0],
                                size=min(N_NOISE_PER_HEAD, bg_embs_full.shape[0]),
                                replace=False)
        bg_embs = bg_embs_full[bg_idx]
        head = train_fewshot_head(enroll_embs, unknown_embs, bg_embs)

        # Score everything
        head.eval()
        def _score(embs):
            if embs.shape[0] == 0:
                return np.array([])
            with torch.no_grad():
                logits = head(torch.from_numpy(embs).to(device))
            return F.softmax(logits, dim=1)[:, TARGET_LBL].cpu().numpy()

        holdout_score   = float(_score(holdout_emb[None])[0]) if holdout_emb is not None else float('nan')
        cross_scores    = _score(cross_embs)
        mswc_scores     = _score(mswc_distractor_embs)

        mean_cross = float(np.mean(cross_scores)) if cross_scores.size else float('nan')
        mean_mswc  = float(np.mean(mswc_scores))
        passed     = (not math.isnan(holdout_score)
                      and holdout_score > 0.5
                      and (math.isnan(mean_cross) or mean_cross < 0.5)
                      and mean_mswc < 0.5)

        novel_results.append({
            'lang': lang, 'keyword': keyword,
            'n_enroll': int(enroll_embs.shape[0]),
            'holdout_score': holdout_score,
            'mean_cross_novel_score': mean_cross,
            'mean_mswc_score': mean_mswc,
            'passed': bool(passed),
        })
        print(f'  {lang}/{keyword:12s}  n_enroll={enroll_embs.shape[0]}  '
              f'holdout={holdout_score:.3f}  '
              f'cross_novel={mean_cross:.3f}  '
              f'MSWC={mean_mswc:.3f}  '
              f'{"PASS" if passed else "fail"}')

    # Persist + summary table
    if novel_results:
        NOVEL_RESULTS_PATH.write_text(json.dumps(novel_results, indent=2,
                                                ensure_ascii=False), encoding='utf-8')
        print(f'\nSaved {len(novel_results)} novel-keyword results -> {NOVEL_RESULTS_PATH}')

        print(f'\n{"lang/keyword":24s} {"holdout":>9s} {"cross":>9s} {"MSWC":>9s} {"verdict":>9s}')
        n_pass = 0
        for r in novel_results:
            n_pass += int(r['passed'])
            print(f'{r["lang"]}/{r["keyword"]:18s} '
                  f'{r["holdout_score"]:9.3f} '
                  f'{r["mean_cross_novel_score"]:9.3f} '
                  f'{r["mean_mswc_score"]:9.3f} '
                  f'{"PASS" if r["passed"] else "fail":>9s}')
        print(f'\nPass rate: {n_pass}/{len(novel_results)} '
              f'({100*n_pass/max(len(novel_results),1):.0f}%)')

        # Optional bar-chart visualisation
        fig, ax = plt.subplots(figsize=(10, 4))
        labels = [f'{r["lang"]}/{r["keyword"]}' for r in novel_results]
        x = np.arange(len(labels))
        w = 0.27
        ax.bar(x - w, [r['holdout_score'] for r in novel_results], w,
               label='Held-out TTS (positive, expect high)', color='tab:green')
        ax.bar(x,     [r['mean_cross_novel_score'] for r in novel_results], w,
               label='Other novel TTS (negative, expect low)', color='tab:orange')
        ax.bar(x + w, [r['mean_mswc_score'] for r in novel_results], w,
               label='Random MSWC (negative, expect low)', color='tab:blue')
        ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Decision threshold 0.5')
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
        ax.set_ylabel('P(target)')
        ax.set_ylim(0, 1.05)
        ax.set_title(f'Novel-keyword TTS-enrolled heads ({n_pass}/{len(novel_results)} pass)')
        ax.legend(loc='best', fontsize=8)
        plt.tight_layout()
        plt.savefig(CACHE_DIR / 'novel_keyword_demo.png', dpi=150)
        plt.show()

except ImportError:
    print('edge-tts not installed.  Run: !pip install -q edge-tts nest_asyncio')
except Exception as exc:
    print(f'Section 8c failed (non-fatal): {type(exc).__name__}: {exc}')


---
## Section 9 — Save & summary

We save:
- The trained embedding (float32 PyTorch state dict) — input to `multilingual_kws_export.ipynb`
- The inventory + class-to-index mapping (so the export notebook reconstructs the architecture)
- The 5-shot results JSON (for the report)
- Plots: training curves, t-SNE, 5-shot ROC

A separate `multilingual_kws_export.ipynb` notebook will pick up the checkpoint and produce the int8 ONNX deployment artifact + ESP32 budget table.

In [ ]:
# ── Final checkpoint bundle ───────────────────────────────────────────────────
bundle_path = CACHE_DIR / 'embedding_final.pt'
torch.save({
    'state_dict':   model.state_dict(),
    'n_classes':    N_CLASSES,
    'embed_dim':    EMBEDDING_DIM,
    'class_to_idx': {f'{k[0]}|{k[1]}': v for k, v in class_to_idx.items()},
    'inventory':    inventory,
    'history':      history,
    'fewshot_summary': {
        'overall_mean_f1':   float(np.mean(all_f1)),
        'overall_mean_auc':  float(np.mean(all_aucs)),
        'per_lang':          [
            {'lang': l, 'n_kw': n, 'mean_f1': mf, 'median_f1': medf, 'mean_auc': ma}
            for l, n, mf, medf, ma in per_lang_summary
        ],
    },
    'config': {
        'SAMPLE_RATE': SAMPLE_RATE, 'N_MELS': N_MELS,
        'N_FFT': N_FFT, 'WIN_LENGTH': WIN_LENGTH, 'HOP_LENGTH': HOP_LENGTH,
        'TOP_K_PER_LANG': TOP_K_PER_LANG,
        'SAMPLES_PER_CLASS': SAMPLES_PER_CLASS,
        'N_SHOT': N_SHOT, 'F1_THRESHOLD': F1_THRESHOLD,
    },
}, bundle_path)
print(f'Checkpoint bundle saved: {bundle_path}  ({bundle_path.stat().st_size/1e6:.1f} MB float32)')

# Summary of all artefacts
artefacts = [
    CACHE_DIR / 'keyword_inventory.json',
    CACHE_DIR / 'embedding_best.pt',
    CACHE_DIR / 'embedding_final.pt',
    CACHE_DIR / 'training_curves.png',
    CACHE_DIR / 'tsne_languages.png',
    CACHE_DIR / 'fewshot_results.png',
    CACHE_DIR / 'fewshot_results.json',
]
print('\nArtefacts produced:')
for p in artefacts:
    if p.exists():
        print(f'  {str(p):60s} {p.stat().st_size/1e3:8.1f} KB')
    else:
        print(f'  {str(p):60s}  (not generated)')

# Quick text summary
print('\n' + '=' * 70)
print('SUMMARY')
print('=' * 70)
print(f'Backbone params         : {model.count_params():,}')
print(f'Embedding dim           : {EMBEDDING_DIM}')
print(f'Languages               : {len(LANGUAGES)}')
print(f'Training classes        : {N_CLASSES} (noise excluded)')
print(f'Best val top-1 accuracy : {max(history["val_acc"]):.3f}')
print(f'5-shot mean F1 @ {F1_THRESHOLD}    : {np.mean(all_f1):.3f}  '
      f'(Mazumder 11M EffNet baseline: 0.75)')
print(f'5-shot mean ROC-AUC     : {np.mean(all_aucs):.3f}')
print('=' * 70)
print('\nNext step: run multilingual_kws_export.ipynb to produce the int8 ONNX')
print('deployment artifact and the ESP32 budget table.')